In [1]:
!pip install sentence-transformers python-docx PyPDF2 pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.3 MB/s eta 0:00:00


In [2]:
import pandas as pd
df = pd.read_csv("fake_job_postings.csv")  # dosya adı
print(df.columns.tolist())
print(df.shape)
print(df.head(2))

['job_id', 'title', 'location', 'department', 'salary_range', 'company_profile', 'description', 'requirements', 'benefits', 'telecommuting', 'has_company_logo', 'has_questions', 'employment_type', 'required_experience', 'required_education', 'industry', 'function', 'fraudulent']
(17880, 18)
   job_id                                      title          location  \
0       1                           Marketing Intern  US, NY, New York   
1       2  Customer Service - Cloud Video Production    NZ, , Auckland   

  department salary_range                                    company_profile  \
0  Marketing          NaN  We're Food52, and we've created a groundbreaki...   
1    Success          NaN  90 Seconds, the worlds Cloud Video Production ...   

                                         description  \
0  Food52, a fast-growing, James Beard Award-winn...   
1  Organised - Focused - Vibrant - Awesome!Do you...   

                                        requirements  \
0  Experience with 

In [3]:
import pandas as pd
import PyPDF2
from docx import Document
from google.colab import files
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

In [4]:
df = pd.read_csv("fake_job_postings.csv")

# Sadece gerçek ilanları kullan (fraudulent=0), description+requirements birleştir
df = df[df["fraudulent"] == 0].copy()
df["full_text"] = df["title"].fillna("") + " " + \
                  df["description"].fillna("") + " " + \
                  df["requirements"].fillna("")
df = df[df["full_text"].str.len() > 100].reset_index(drop=True)

print(f"Kullanılabilir ilan sayısı: {len(df)}")

Kullanılabilir ilan sayısı: 16996


In [10]:
SKILL_LIST = [
    # CV'dekiler
    "python", "c", "c#", "c++", "java", "javascript", "sql",
    "ms sql", "git", "github", "linux", "ubuntu", "vs code",
    "visual studio", "esp32", "freertos", "uart", "i2c", "spi",
    "embedded systems", "sensor", "iot", "cybersecurity",
    "object-oriented programming", "oop", "database",
    # Genel teknik
    "machine learning", "deep learning", "nlp", "tensorflow",
    "pytorch", "docker", "flask", "django", "react", "aws",
    "html", "css", "agile", "scrum", "communication",
    "teamwork", "leadership", "project management",
    "real-time", "networking", "freertos", "rtos",
    "microcontroller", "firmware", "driver"
]

def extract_skills(text):
    text = text.lower()
    return list(set([s for s in SKILL_LIST if s in text]))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s]', '', text)
    return text.strip()

In [11]:
def extract_text_from_pdf(path):
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

def extract_text_from_docx(path):
    doc = Document(path)
    return " ".join([p.text for p in doc.paragraphs])

print("CV dosyanızı yükleyin (PDF veya DOCX):")
uploaded = files.upload()
filename = list(uploaded.keys())[0]

if filename.endswith(".pdf"):
    cv_text = extract_text_from_pdf(filename)
elif filename.endswith(".docx"):
    cv_text = extract_text_from_docx(filename)
else:
    cv_text = ""
    print("Hata: Sadece PDF veya DOCX desteklenir!")

cv_skills = extract_skills(cv_text)
print(f"\n✅ CV'de tespit edilen beceriler: {cv_skills}")

CV dosyanızı yükleyin (PDF veya DOCX):


Saving Evla_Karagoz_Embedded_CV.pdf to Evla_Karagoz_Embedded_CV (1).pdf

✅ CV'de tespit edilen beceriler: ['object-oriented programming', 'visual studio', 'iot', 'linux', 'vs code', 'real-time', 'ubuntu', 'driver', 'rtos', 'c#', 'c', 'sql', 'cybersecurity', 'teamwork', 'esp32', 'python', 'i2c', 'embedded systems', 'ms sql', 'git', 'uart', 'sensor', 'spi', 'networking', 'github', 'freertos', 'database', 'communication']


In [14]:
print("Model yükleniyor...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# İlk 500 ilan (hız için, tüm dataset çok uzun sürer)
sample_df = df.sample(500, random_state=42).reset_index(drop=True)

print("Embeddingler hesaplanıyor...")
cv_embedding = model.encode([preprocess(cv_text)])
job_embeddings = model.encode(sample_df["full_text"].tolist(), show_progress_bar=True)

scores = cosine_similarity(cv_embedding, job_embeddings)[0]
sample_df["uyum_skoru"] = scores
sample_df = sample_df.sort_values("uyum_skoru", ascending=False).reset_index(drop=True)

print("✅ Tamamlandı!")

Model yükleniyor...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddingler hesaplanıyor...


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

✅ Tamamlandı!


In [15]:
def analiz_raporu(cv_text, cv_skills, sample_df):
    print("=" * 65)
    print("        📋 SKİLLSYNC — CV ANALİZ RAPORU")
    print("=" * 65)

    best = sample_df.iloc[0]

    # Genel uyum
    print(f"\n🏆 EN YÜKSEK UYUMLU İLAN")
    print(f"   Pozisyon  : {best['title']}")
    print(f"   Lokasyon  : {best['location']}")
    print(f"   Sektör    : {best['industry'] if pd.notna(best['industry']) else 'Belirtilmemiş'}")
    print(f"   Uyum Skoru: %{best['uyum_skoru']*100:.1f}")

    # Skill gap
    best_skills = extract_skills(str(best["full_text"]))
    matched = [s for s in best_skills if s in cv_skills]
    missing = [s for s in best_skills if s not in cv_skills]

    print(f"\n✅ CV'deki Eşleşen Beceriler : {matched if matched else 'Tespit edilemedi'}")
    print(f"❌ CV'de Eksik Beceriler     : {missing if missing else 'Eksik yok'}")

    if missing:
        tahmini = min(best['uyum_skoru'] + len(missing) * 0.04, 1.0)
        print(f"\n💡 Bu eksik beceriler kazanılsaydı tahmini uyum: %{tahmini*100:.1f}")

    # Alan istatistikleri
    print(f"\n📊 ALAN İSTATİSTİKLERİ")
    print(f"   Analiz edilen ilan sayısı     : {len(sample_df)}")
    print(f"   %50 üzeri uyumlu ilan sayısı : {(sample_df['uyum_skoru'] > 0.5).sum()}")
    print(f"   %70 üzeri uyumlu ilan sayısı : {(sample_df['uyum_skoru'] > 0.7).sum()}")
    print(f"   Ortalama uyum skoru           : %{sample_df['uyum_skoru'].mean()*100:.1f}")

    # Sektör dağılımı
    print(f"\n🏭 ALANINIZDAKI EN YAYGIN SEKTÖRLER (Top 5):")
    top_sektorler = sample_df["industry"].value_counts().head(5)
    for sektor, sayi in top_sektorler.items():
        print(f"   {sektor}: {sayi} ilan")

    # Top 5 ilan
    print(f"\n🔝 SİZE EN UYGUN İLK 5 İLAN:")
    for i, row in sample_df.head(5).iterrows():
        print(f"   {i+1}. {row['title']} — %{row['uyum_skoru']*100:.1f} uyum")

    print("\n" + "=" * 65)

analiz_raporu(cv_text, cv_skills, sample_df)

        📋 SKİLLSYNC — CV ANALİZ RAPORU

🏆 EN YÜKSEK UYUMLU İLAN
   Pozisyon  : Programmer
   Lokasyon  : GR, B, Thessaloniki
   Sektör    : Research
   Uyum Skoru: %53.9

✅ CV'deki Eşleşen Beceriler : ['c', 'database']
❌ CV'de Eksik Beceriler     : Eksik yok

📊 ALAN İSTATİSTİKLERİ
   Analiz edilen ilan sayısı     : 500
   %50 üzeri uyumlu ilan sayısı : 2
   %70 üzeri uyumlu ilan sayısı : 0
   Ortalama uyum skoru           : %19.7

🏭 ALANINIZDAKI EN YAYGIN SEKTÖRLER (Top 5):
   Information Technology and Services: 56 ilan
   Computer Software: 48 ilan
   Internet: 36 ilan
   Marketing and Advertising: 23 ilan
   Education Management: 23 ilan

🔝 SİZE EN UYGUN İLK 5 İLAN:
   1. Programmer — %53.9 uyum
   2. Operations Contract — %50.9 uyum
   3. DW Database Administrator — %49.1 uyum
   4. Security Data Architect — %46.4 uyum
   5. SR Programmer Analyst — %42.2 uyum



In [9]:
print(cv_text[:1000])

Programming: C, Python, C#
Embedded Systems: ESP32 (ESP-IDF), FreeRTOS (basics), sensor integration
Communication Protocols: UART, I2C, SPI
Tools & Technologies: Git, GitHub, Linux (Ubuntu), VS Code, Visual Studio, MS SQL
Concepts: Object-Oriented Programming, basic database design
İstanbul Gedik Üniversitesi | Computer Engineering  10/2023 - Present
Linux Essentials – Cisco Networking Academy (2026)
Introduction to IoT – Cisco Networking Academy (2026)
Industrial Networking Essentials – Cisco Networking Academy (2026)
Introduction to Cybersecurity – Cisco Networking Academy (2026)
TEKNOFEST 2024 Achievement Certificate – TEKNOFEST (2024)Embedded Systems Intern 
Fluxion Lab – TÜBİTAK Marmara Technokent | Jun 2025 – Aug 2025
Optimized task scheduling and resource management on ESP32-S3 using FreeRTOS, improving system efficiency and
responsiveness
Developed low-level drivers for UART, I2C, and SPI to ensure reliable sensor data acquisition
Designed real-time data processing algorithms u